In [5]:
import matplotlib.pyplot as plt
import numpy as np
import os
from matplotlib.ticker import MultipleLocator
from matplotlib.colors import LogNorm, LinearSegmentedColormap
from mpl_toolkits.axes_grid1 import make_axes_locatable

def clarke_error_grid_normalized_by_dataset(ref_values, 
                                            pred_values, 
                                            title_string,
                                            plot_dir='Final_test',
                                            vmin_manual=None, 
                                            vmax_manual=None,
                                            show_plot=False,
                                            minimum_sensor_reading=40,
                                            maximum_sensor_reading=500):
    
    if not os.path.exists(plot_dir):
        os.makedirs(plot_dir)
    
    # Saturate values
    ref_values = ref_values.clip(0, 500)
    pred_values = pred_values.clip(0, 500)
    
    # Checking to see if the lengths of the reference and prediction arrays are the same
    # Arrays compactos para bajar RAM
    ref_values = np.asarray(ref_values, dtype=np.float32)
    pred_values = np.asarray(pred_values, dtype=np.float32)

    assert (len(ref_values) == len(
        pred_values)), "Unequal number of values (reference : {0}) (prediction : {1}).".format(len(ref_values),
                                                                                               len(pred_values))

    # Checks to see if the values are within the normal sensor measurement range, otherwise it gives a warning
    # Reference values
    if min(ref_values) < minimum_sensor_reading:
        print(
            f'Input Warning: the minimum reference value ({min(ref_values):.2f}) is below the limit of sensor design (Inferior limit: {minimum_sensor_reading} mg/dL).')
        number_ref_values_low_limit = sum(value < minimum_sensor_reading for value in ref_values)
        print(f"Number of reference values below the limit: {number_ref_values_low_limit}")
        print()

    if max(ref_values) > maximum_sensor_reading:
        print(
            f'Input Warning: the maximum reference value ({max(ref_values):.2f}) is above the limit of sensor design (Superior limit: {maximum_sensor_reading} mg/dL).')
        number_ref_values_up_limit = sum(value > maximum_sensor_reading for value in ref_values)
        print(f"Number of reference values above the limit: {number_ref_values_up_limit}")
        print()

    # Predicted values
    min_ref_value = min(pred_values)
    min_ref_value = round(min_ref_value, 2)
    number_pred_values_low_limit = 0
    if min_ref_value < minimum_sensor_reading:
        print(
            f'Input Warning: the minimum predicted value ({min_ref_value:.2f}) is below the minimum input value (Inferior limit: {minimum_sensor_reading} mg/dL).')
        number_pred_values_low_limit = sum(value < minimum_sensor_reading for value in pred_values)
        # list_of_values_low_limit = [value for value in pred_values if value < minimum_sensor_reading]
        print(f"Number of predicted values below the limit: {number_pred_values_low_limit}")
        print()

    max_ref_value = max(pred_values)
    max_ref_value = round(max_ref_value, 2)
    number_pred_values_up_limit = 0
    if max_ref_value > maximum_sensor_reading:
        print(
            f'Input Warning: the maximum predicted value ({max_ref_value:.2f}) is above the maximum input value (Superior limit: {maximum_sensor_reading} mg/dL).')
        number_pred_values_up_limit = sum(value > maximum_sensor_reading for value in pred_values)
        # list_of_values_up_limit = [value for value in pred_values if value > maximum_sensor_reading]
        print(f"Number of predicted values above the limit: {number_pred_values_up_limit}")
        print()

    # list of out of range values
    out_range_values = [min_ref_value, number_pred_values_low_limit, max_ref_value, number_pred_values_up_limit]

    if show_plot:
        # Plot start
        plt.clf()
        fig, ax = plt.subplots() 
    
        # --- CONFIGURACIÓN DE BINS ---
        n_bins = 500
        my_range = [[0, 500], [0, 500]]
        my_bins = [n_bins, n_bins]
    
        # --- CÁLCULO DE LÍMITES (AUTOMÁTICO O MANUAL) ---
        # Si nos pasan los límites (vmin_manual), los usamos. Si no, calculamos.
        if vmin_manual is not None and vmax_manual is not None:
            vmin_global = vmin_manual
            vmax_global = vmax_manual
        else:
            # Cálculo local estándar si no se comparte escala
            h_temp = np.histogram2d(ref_values, pred_values, bins=my_bins, range=my_range)
            # El count mínimo siempre es 1 (para log), el máximo es el max del histograma
            vmin_global = 1 
            vmax_global = np.max(h_temp[0]) if np.max(h_temp[0]) > 0 else 10
    
        # Normalización compartida
        global_norm = LogNorm(vmin=vmin_global, vmax=vmax_global)
    
        # --- DEFINICIÓN DE COLORES ---
        cmap_tbr2 = LinearSegmentedColormap.from_list("custom_indian", ["#ffffff", "indianred", "#8B3A3A"]) 
        cmap_tbr1 = LinearSegmentedColormap.from_list("custom_red", ["#ffffff", "red", "#8B0000"])
        cmap_tir  = LinearSegmentedColormap.from_list("custom_lime", ["#ffffff", "limegreen", "#006400"])
        cmap_tar1 = LinearSegmentedColormap.from_list("custom_yellow", ["#ffffff", "yellow", "#808000"])
        cmap_tar2 = LinearSegmentedColormap.from_list("custom_orange", ["#ffffff", "orange", "#8B4513"])
        cmaps_ordered = [cmap_tbr2, cmap_tbr1, cmap_tir, cmap_tar1, cmap_tar2]
    
        # --- DIBUJO ---
        ranges = [
            (ref_values < 54, cmap_tbr2),
            ((ref_values >= 54) & (ref_values < 70), cmap_tbr1),
            ((ref_values >= 70) & (ref_values <= 180), cmap_tir),
            ((ref_values > 180) & (ref_values <= 250), cmap_tar1),
            (ref_values > 250, cmap_tar2)
        ]
    
        for mask, cmap in ranges:
            if np.any(mask):
                # Importante: pasamos norm=global_norm para respetar la escala compartida
                ax.hist2d(ref_values[mask], pred_values[mask],
                          bins=my_bins, range=my_range, cmap=cmap, cmin=1, norm=global_norm, zorder=2)
    
        # --- BARRA DE LEYENDA (Striped) ---
        divider = make_axes_locatable(ax)
        cax = divider.append_axes("right", size="5%", pad=0.2) 
    
        n_steps = 200
        Y_bounds = np.logspace(np.log10(vmin_global), np.log10(vmax_global), n_steps + 1)
        Z_values = np.logspace(np.log10(vmin_global), np.log10(vmax_global), n_steps)
        Z_slice = Z_values.reshape(-1, 1)
    
        for i in range(5):
            X_bounds = np.array([i, i+1])
            X_mesh, Y_mesh = np.meshgrid(X_bounds, Y_bounds)
            cax.pcolormesh(X_mesh, Y_mesh, Z_slice, cmap=cmaps_ordered[i], norm=global_norm, shading='flat')
    
        cax.set_yscale('log')
        cax.set_xlim(0, 5)
        cax.yaxis.tick_right()
        # cax.yaxis.set_label_position("right")
        # cax.set_ylabel('Point Density (Log Scale)', rotation=90)
        cax.set_xticks([]) # Sin etiquetas abajo
    
        # --- ESTÉTICA FINAL (Inglés) ---
        lp = {'c': 'black', 'linewidth': 0.8, 'zorder': 10, 'alpha': 0.5}
        # lp = {'c': 'black', 'linewidth': 1}
        ax.plot([0, 500], [0, 500], ':', **lp)
        
        ax.plot([0, 175/3], [70, 70], '-', **lp)
        ax.plot([175/3, 500/1.2], [70, 500], '-', **lp)
        
        ax.plot([70, 70], [84, 500], '-', **lp)
        
        ax.plot([0, 70], [180, 180], '-', **lp)
        ax.plot([70, 390], [180, 500], '-', **lp)
        ax.plot([70, 70], [0, 56], '-', **lp)
        
        ax.plot([70, 500], [56, 400], '-', **lp)
        
        ax.plot([180, 180], [0, 70], '-', **lp)
        ax.plot([180, 500], [70, 70], '-', **lp)
        ax.plot([240, 240], [70, 180], '-', **lp)
        ax.plot([240, 500], [180, 180], '-', **lp)
        ax.plot([130, 180], [0, 70], '-', **lp)
    
        zones_ceg = [(380,440,"A"),(440,380,"A"),(170,100,"B"),(100,170,"B"),
                     (150,400,"C"),(155,15,"C"),(30,120,"D"),(370,120,"D"),(30,340,"E"),(370,15,"E")]
        for x, y, txt in zones_ceg:
            ax.text(x, y, txt, fontsize=12, fontweight='bold', color='black', alpha=0.6, zorder=11)
            # ax.text(x, y, txt, fontsize=12) #, fontweight='bold', color='black', alpha=0.6, zorder=11)
    
        ax.set_xlabel("Reference Concentration (mg/dl)")
        ax.set_ylabel("Prediction Concentration (mg/dl)")
        ax.set_xticks(np.arange(0, 551, 50)); ax.set_yticks(np.arange(0, 551, 50))
        ax.xaxis.set_minor_locator(MultipleLocator(10)); ax.yaxis.set_minor_locator(MultipleLocator(10))
        ax.set_xlim([0, 501]); ax.set_ylim([-1, 500])
        ax.set_aspect('equal')
    
        filename = f"{plot_dir}/CEG_SharedScale_{title_string}.png"
        plt.savefig(filename, dpi=900, bbox_inches='tight')
        plt.close()
        print(f"Generado: {filename}")
        # Plot End
    
    # Statistics from the data
    zone = [0] * 5
    for i in range(len(ref_values)):
        # ---------------------------------------------------------------------------------
        # ZONE A (Clinical Accurate)
        # ---------------------------------------------------------------------------------
        if ((ref_values[i] < 70 and pred_values[i] < 70)  # Condition 1 for Zone A (LEFT BOTTOM)
                or
                (pred_values[i] <= 1.2 * ref_values[i] and pred_values[i] >= 0.8 * ref_values[
                    i])):  # Condition 2 for Zone A (RIGHT DIAGONAL)
            zone[0] += 1

        # EXPLANATION:
        # (ref & pred < 70) - EXCLUSIVE: True to the clinical definition in original Clarke paper text.
        # (<= 1.2*ref and >= 0.8*ref) - INCLUSIVE: Captures the ±20% range in original Clarke paper text.

        # End of Zone A conditions --------------------

        # ---------------------------------------------------------------------------------
        # ZONE E (Erroneous Treatment)
        # ---------------------------------------------------------------------------------
        elif ((ref_values[i] > 180 and pred_values[i] < 70)  # Condition 1 for Zone E (BOTTOM RIGHT)
              or
              (ref_values[i] < 70 and pred_values[i] > 180)):  # Condition 2 for Zone E (TOP LEFT)

            zone[4] += 1  # Zone E

        # EXPLANATION:
        # (< 70 and > 180) - EXCLUSIVE: True to the original Clarke paper text.
        # A real value of 70 or 180 is considered "in range," not "hypo" or "hyper."
        # Therefore, (180, 69) or (70, 181) are not Zone E.

        # End of Zone E conditions --------------------

        # ---------------------------------------------------------------------------------
        # ZONE C (Overcorrection)
        # ---------------------------------------------------------------------------------
        elif (((ref_values[i] >= 70 and ref_values[i] <= 290) and pred_values[i] >= ref_values[
            i] + 110)  # Condition 1 for Zone C (TOP)
              or
              ((ref_values[i] >= 130 and ref_values[i] <= 180) and (
                      pred_values[i] <= (7 / 5) * ref_values[i] - 182))):  # Condition 2 for Zone C (BOTTOM)

            zone[2] += 1  # Zone C

        # EXPLANATION:
        # Community consensus to use inclusive limits for Zone C conditions

        # End of Zone C conditions --------------------

        # ---------------------------------------------------------------------------------
        # ZONE D (Failure to Detect)
        # #---------------------------------------------------------------------------------
        elif ((ref_values[i] > 240 and (
                pred_values[i] >= 70 and pred_values[i] <= 180))  # Condition 1 for Zone D (RIGHT) # DUDA
              or
              (ref_values[i] <= 175 / 3 and pred_values[i] <= 180 and pred_values[
                  i] >= 70)  # Condition 2 for Zone D (LEFT)
              or
              ((ref_values[i] >= 175 / 3 and ref_values[i] < 70) and pred_values[i] >= (6 / 5) * ref_values[
                  i])):  # Condition 3 for Zone D (LEFT)

            zone[3] += 1

            # EXPLANATION OF THE CHANGES:
            # The original code used (ref <= 70) in condition D3.
            # This caused a point like (70, 90) to be classified as Zone D.
            # But (70, 90) is clinically a Zone B error,
            # since 70 is the edge of the target range, not hypoglycemia.
            # By changing it to (ref < 70), (70, 90) fails this condition and
            # correctly falls through to the 'else' (Zone B).
            #
            # Likewise, condition D1 was changed from (ref >= 240) to (ref > 240).
            # This change maintains consistency with the original Clarke paper text,
            # which textually defines this specific clinical failure as (ref > 240).
            #
            # The other prediction value limits (>= 70, <= 180) in D1 and D2 are correct because
            # they define the "target range" that the meter erroneously predicts.

            # End of Zone D conditions --------------------

        else:
            zone[1] += 1  # Zone B

    return zone, out_range_values

In [7]:
import pandas as pd
df = pd.read_parquet('c:/Users/Ciro/C/_UGR_PDI/RELIEF-T1D/Glucose_prediction_2026/prediction_vectors/t1d/df_test_results_vectors_RMSE_Linear_H4.parquet')
clarke_error_grid_normalized_by_dataset(ref_values=df['y_test'],
                                 pred_values=df['y_predict'],
                                 title_string='TEst',)

Generado: Final_test/CEG_SharedScale_TEst.png


<Figure size 640x480 with 0 Axes>